# PyTorch Lightning

**Domain:** AI/ML Tooling  ·  **recommended addition**  ·  **runnable:** yes

A refresher on **PyTorch Lightning** — the framework that takes your raw PyTorch model and removes the engineering boilerplate (training loops, device placement, distributed training, checkpointing, logging) without hiding the model itself.

## 1. What & Why

Plain PyTorch makes you hand-write the same training loop every project: zero the gradients, forward, compute loss, backward, step the optimizer, move tensors to the GPU, accumulate metrics, save checkpoints, switch `model.train()`/`model.eval()`, and bolt on multi-GPU/AMP later. It's repetitive, easy to get subtly wrong (forgotten `optimizer.zero_grad()`, a tensor left on CPU), and hard to scale.

**PyTorch Lightning** keeps your model and math as ordinary PyTorch but moves the *engineering* into a reusable, tested engine. You subclass `LightningModule` and fill in a few hooks — `training_step`, `configure_optimizers`, etc. — and a `Trainer` runs the loop. The same code runs unchanged on CPU, one GPU, many GPUs, or TPUs; you only change `Trainer` flags.

**Reach for it when** you want production-grade training infrastructure (distributed training, mixed precision, checkpointing, early stopping, experiment logging) without writing or debugging it yourself, and you want your research code to stay readable and reproducible. **Skip it when** you're doing a tiny one-off script where a 10-line loop is clearer, or you need a highly custom loop that fights the framework's structure (though Lightning's hooks and `manual_optimization` cover most of those cases too).

## 2. Mental Model

**Lightning splits your code into two halves: the *what* (your `LightningModule`) and the *how* (the `Trainer`).**

```
LightningModule  =  model + the science                Trainer = the engineering
------------------------------------                   --------------------------------
__init__            (define layers)                    loops over epochs/batches
forward             (inference)                         calls .backward(), optimizer.step()
training_step       (loss for one batch)                moves data/model to the device
validation_step     (val metric for one batch)          mixed precision, gradient clipping
configure_optimizers(optimizer + scheduler)             multi-GPU / multi-node
                                                        checkpointing, early stopping, logging
```

You write the left column once. The right column is the same engine everyone uses, so you never re-implement (or re-debug) `zero_grad → forward → loss → backward → step`. Crucially, **a `LightningModule` *is* an `nn.Module`** — `forward`, `state_dict`, and `.to(device)` all still work, so you can drop a trained Lightning model into plain PyTorch for inference. Lightning organizes your code; it doesn't replace PyTorch.

## 3. Key Concepts

| Concept | What it is |
|---|---|
| **`LightningModule`** | Your model + the training/validation/test logic, packaged as `nn.Module` subclass with named hooks. |
| **`training_step(batch, idx)`** | Compute and **return the loss** for one batch. Lightning does `backward`/`step` for you. |
| **`validation_step` / `test_step`** | Same idea for eval; no gradients, `model.eval()` set automatically. Usually log a metric. |
| **`configure_optimizers`** | Return the optimizer(s) and (optionally) LR scheduler(s). |
| **`Trainer`** | The execution engine. Flags like `max_epochs`, `accelerator`, `devices`, `precision` control everything. |
| **`self.log(name, value)`** | Records a metric; Lightning aggregates it across batches/devices and hands it to loggers/progress bar. |
| **`LightningDataModule`** | Optional bundle of `train/val/test` `DataLoader`s + download/setup logic for reproducible data. |
| **Callbacks** | Pluggable behavior at lifecycle points — `ModelCheckpoint`, `EarlyStopping`, LR monitors — without touching the loop. |
| **`seed_everything`** | One call to seed Python/NumPy/Torch for reproducibility. |
| **Hooks** | ~40 override points (`on_train_epoch_end`, `on_after_backward`, …) for custom behavior without owning the loop. |

## 4. Setup

Lightning is pure PyTorch under the hood, so you need `torch` plus the framework. `torchmetrics` (same authors) gives device- and distributed-correct metrics.

```bash
pip install lightning torchmetrics      # the modern unified package
# import lightning as L
```

Note the two import names that both still work and cause confusion (see Gotchas):

- `import lightning as L` — the current **unified** package (`lightning.pytorch`).
- `import pytorch_lightning as pl` — the older standalone package, still maintained, same API.

Everything below runs on CPU in a few seconds.

In [1]:
# If Lightning isn't installed in this kernel, uncomment:
# %pip install -q lightning torchmetrics
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import lightning as L

L.seed_everything(0, verbose=False)   # reproducible Python/NumPy/Torch RNG
print(f"torch     {torch.__version__}")
print(f"lightning {L.__version__}")
print(f"accelerator available: cpu (this notebook is CPU-only by design)")

/Users/danieldekerlegand/Development/ai-tutor/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch     2.12.1
lightning 2.6.5
accelerator available: cpu (this notebook is CPU-only by design)


## 5. Worked Examples

Two tiny, CPU-friendly examples: (1) a full `LightningModule` + `Trainer` training loop on synthetic data, and (2) adding a validation loop, `torchmetrics` accuracy, and callbacks (`EarlyStopping`, `ModelCheckpoint`) — all without touching the training loop.

### 5.1 A complete `LightningModule` + `Trainer`

A 2-layer MLP that classifies points as inside vs. outside a circle. Notice what's *absent*: no manual `zero_grad`/`backward`/`step`, no `.to(device)`, no epoch loop. You write `training_step` (return the loss) and `configure_optimizers`; the `Trainer` runs everything.

In [2]:
# Toy task: is the 2D point outside a circle of radius^2 = 1.5?
X = torch.randn(512, 2)
y = (X.pow(2).sum(1) > 1.5).long()
train_ds = TensorDataset(X, y)
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)

class LitMLP(L.LightningModule):
    def __init__(self, din=2, dhidden=16, dout=2, lr=1e-2):
        super().__init__()
        self.save_hyperparameters()           # stored in checkpoints, shown in self.hparams
        self.net = nn.Sequential(
            nn.Linear(din, dhidden), nn.ReLU(),
            nn.Linear(dhidden, dout),
        )

    def forward(self, x):                      # used for inference; it's still an nn.Module
        return self.net(x)

    def training_step(self, batch, batch_idx):
        x, target = batch
        logits = self(x)
        loss = F.cross_entropy(logits, target)
        self.log("train_loss", loss, prog_bar=False)   # aggregated + logged for you
        return loss                             # Lightning does backward + optimizer.step

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

model = LitMLP()
trainer = L.Trainer(
    max_epochs=15, accelerator="cpu",
    enable_progress_bar=False, logger=False, enable_checkpointing=False,
    enable_model_summary=False,
)
trainer.fit(model, train_dl)

# It's still plain PyTorch for inference:
model.eval()
with torch.no_grad():
    acc = (model(X).argmax(1) == y).float().mean()
print(f"final training-set accuracy: {acc:.3f}")

GPU available: True (mps), used: False


TPU available: False, using: 0 TPU cores


/Users/danieldekerlegand/Development/ai-tutor/.venv/lib/python3.13/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


/Users/danieldekerlegand/Development/ai-tutor/.venv/lib/python3.13/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/danieldekerlegand/Development/ai-tutor/.venv/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.
`Trainer.fit` stopped: `max_epochs=15` reached.


final training-set accuracy: 0.988


### 5.2 Validation, metrics, and callbacks

Real training needs a validation loop, a proper metric, and automation. Here we add a `validation_step` with a `torchmetrics` accuracy, then hand the `Trainer` two callbacks — `EarlyStopping` (stop when val loss plateaus) and `ModelCheckpoint` (save the best model). None of this touches the loop; you just plug them in.

In [3]:
import torchmetrics
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

# Train/val split of the same toy data
n_val = 128
val_dl = DataLoader(TensorDataset(X[:n_val], y[:n_val]), batch_size=64)
tr_dl  = DataLoader(TensorDataset(X[n_val:], y[n_val:]), batch_size=64, shuffle=True)

class LitMLP2(LitMLP):
    def __init__(self, **kw):
        super().__init__(**kw)
        self.val_acc = torchmetrics.classification.Accuracy(task="multiclass", num_classes=2)

    def validation_step(self, batch, batch_idx):
        x, target = batch
        logits = self(x)
        loss = F.cross_entropy(logits, target)
        self.val_acc.update(logits, target)
        # log the metric object: Lightning computes/aggregates it at epoch end
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", self.val_acc, prog_bar=True)

ckpt = ModelCheckpoint(monitor="val_loss", mode="min", save_top_k=1, dirpath="/tmp/pl_ckpts")
# stop once val_loss hasn't improved by >= min_delta for `patience` checks
early = EarlyStopping(monitor="val_loss", mode="min", min_delta=1e-3, patience=5)

model2 = LitMLP2(lr=1e-2)
trainer2 = L.Trainer(
    max_epochs=100, accelerator="cpu",
    callbacks=[ckpt, early],
    enable_progress_bar=False, logger=False, enable_model_summary=False,
)
trainer2.fit(model2, tr_dl, val_dl)

print(f"stopped after {trainer2.current_epoch} epochs (EarlyStopping)")
print(f"best val_loss : {ckpt.best_model_score:.4f}")
print(f"best checkpoint: {ckpt.best_model_path.split('/')[-1]}")

# trainer.validate runs the validation loop and returns the logged metrics
metrics = trainer2.validate(model2, val_dl, verbose=False)
print("reloaded val metrics:", {k: round(v, 3) for k, v in metrics[0].items()})

GPU available: True (mps), used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


/Users/danieldekerlegand/Development/ai-tutor/.venv/lib/python3.13/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /private/tmp/pl_ckpts exists and is not empty.
/Users/danieldekerlegand/Development/ai-tutor/.venv/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.


stopped after 90 epochs (EarlyStopping)
best val_loss : 0.0582
best checkpoint: epoch=89-step=540.ckpt
reloaded val metrics: {'val_loss': 0.058, 'val_acc': 0.984}


## 6. Gotchas & Pitfalls

- **Two import names.** `import lightning as L` (modern, unified) and `import pytorch_lightning as pl` (older standalone) are *different packages* with the same API. Mixing them — e.g. a `pytorch_lightning` callback with a `lightning` Trainer — raises confusing type errors. Pick one per project.
- **Don't call `loss.backward()` / `optimizer.step()` yourself** in `training_step` (under the default automatic optimization). Just `return loss`; the Trainer owns the step. If you truly need manual control, set `self.automatic_optimization = False` and use `self.manual_backward(loss)`.
- **Don't move tensors with `.cuda()` / `.to(device)` by hand.** The Trainer places the model and batch on the right device. Create new tensors inside steps with `type_as(x)` or `self.device` so they land on the correct device under multi-GPU.
- **`self.log` semantics differ by stage.** In `training_step` it defaults to `on_step=True`; in `validation_step` to `on_epoch=True`. Be explicit (`on_step=`, `on_epoch=`) when you care, or your "epoch metric" may actually be a per-batch value.
- **Log metric *objects*, not `.compute()` results.** Pass the `torchmetrics` object to `self.log` (as in 5.2) so Lightning aggregates correctly across batches and devices; calling `.compute()` per batch and logging the float silently breaks distributed averaging.
- **`save_hyperparameters()` matters for checkpoints.** Without it, `LitModel.load_from_checkpoint(...)` can't reconstruct your `__init__` args. Call it first thing in `__init__`.
- **`num_sanity_val_steps`** runs 2 validation batches *before* training to catch bugs early — great, but surprising if your `validation_step` has side effects. Set it to 0 to disable.
- **Reproducibility needs more than a seed.** `seed_everything(...)` plus `Trainer(deterministic=True)` for bit-reproducible runs (at some speed cost).

## 7. When to Use vs Alternatives

| Option | Strengths | Weaknesses | Pick it when |
|---|---|---|---|
| **PyTorch Lightning** | Removes loop boilerplate, free multi-GPU/TPU/AMP, callbacks & checkpointing, large ecosystem, model stays plain PyTorch | A structure to learn, can feel heavy for tiny scripts, custom loops need hooks/manual optimization | You want production-grade training without writing the infra; research code that must scale |
| **Raw PyTorch loop** | Total control, nothing hidden, simplest mental model | You re-write & re-debug the loop, device/AMP/distributed are on you | One-off scripts, teaching, or a genuinely non-standard loop |
| **Hugging Face `Trainer` / Accelerate** | Tight integration with `transformers`, great for fine-tuning LLMs; `accelerate` is a lighter "just distribute my loop" layer | `Trainer` is opinionated toward HF models; Accelerate gives less structure than Lightning | Fine-tuning transformer models; or you want distribution without Lightning's structure |
| **fastai** | Very high-level, strong defaults, fast to results | More magic/abstraction, harder to customize deeply | Rapid prototyping, education, applied projects with standard architectures |
| **Keras 3 (multi-backend)** | Clean high-level API, runs on Torch/JAX/TF | Different ecosystem & idioms from PyTorch | You prefer the Keras API or need backend portability |

Rule of thumb: **raw PyTorch for tiny/experimental scripts; Lightning when you want the engineering handled and your code to scale unchanged from laptop to cluster; Hugging Face when your work centers on transformer models.**

## 8. Resources

- **Lightning docs** — https://lightning.ai/docs/pytorch/stable/ — start with "Lightning in 15 minutes".
- **`LightningModule` API** — https://lightning.ai/docs/pytorch/stable/common/lightning_module.html — every hook, in order.
- **`Trainer` flags** — https://lightning.ai/docs/pytorch/stable/common/trainer.html — the full menu of `accelerator`, `precision`, `strategy`, etc.
- **TorchMetrics** — https://lightning.ai/docs/torchmetrics/stable/ — the device-/distributed-correct metrics used in 5.2.
- **Converting plain PyTorch to Lightning** — https://lightning.ai/docs/pytorch/stable/starter/converting.html — the migration checklist if you have an existing loop.